# pisto-gpt-64m - Colab Training (Arabic)

Trains a 68M GPT on **Arabic text only** on a Colab T4 GPU.
Stage 1: Pretrain (1h) → Stage 2: Finetune (1.5h).
Weights save to Google Drive every 30s so a VM recycle never loses progress.

## After a VM death
Just re-run all cells - training **resumes** from the last checkpoint in Drive.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
# 2. Clone the repo from GitHub
import os, subprocess, shutil

REPO    = '/content/pg'
DRIVE_W = '/content/drive/MyDrive/pg-weights'

os.makedirs(DRIVE_W, exist_ok=True)

if not os.path.exists(f'{REPO}/.git'):
    !git clone https://github.com/BayanDrp/pisto-gpt-64m.git {REPO}
else:
    print('Repo already cloned, pulling latest...')
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)

os.makedirs(f'{REPO}/weights', exist_ok=True)

# Restore checkpoints from Drive (resume support)
for f in ['pretrain_best.pt', 'log.jsonl', 'instruct_best.pt', 'instruct_log.jsonl']:
    src = f'{DRIVE_W}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{REPO}/weights/{f}')
        print(f'Restored {f} from Drive')

print('Repo ready at', REPO)

In [ ]:
# 3. Install dependencies
!pip install -q torch datasets tokenizers
print('Deps installed')

In [ ]:
# 4. Retrain tokenizer on Arabic + patch config for Arabic dataset
import json

REPO = '/content/pg'

# --- Retrain BPE tokenizer on Arabic text ---
print('Loading Arabic data for tokenizer training...')
from datasets import load_dataset
arabic_ds = load_dataset(
    'Jr23xd23/ArabicText-Large',
    split='train', streaming=True,
)

arabic_texts = []
for i, s in enumerate(arabic_ds):
    if i >= 100_000:
        break
    text = s.get('text', '')
    if len(text) > 50:
        arabic_texts.append(text)
    if i % 10_000 == 0:
        print(f'  {i:,} docs...')

print(f'Got {len(arabic_texts):,} Arabic docs for tokenizer training')

# Write to temp file for tokenizers library
tok_data = '/tmp/arabic_tok_data.txt'
with open(tok_data, 'w') as f:
    for t in arabic_texts:
        f.write(t.replace(chr(10), ' ') + chr(10))

from tokenizers import Tokenizer, models, pre_tokenizers, trainers
tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
trainer = trainers.BpeTrainer(
    vocab_size=8192,
    special_tokens=['<PAD>', '<BOS>', '<EOS>'],
    min_frequency=2,
)
tok.train([tok_data], trainer)

tok_path = f'{REPO}/config/bpe_tokenizer.json'
tok.save(tok_path)
print(f'Trained Arabic tokenizer saved to {tok_path}')
print(f'Vocab size: {tok.get_vocab_size()}')
print(f'Special tokens: PAD=0, BOS=1, EOS=2')

# --- Patch config/train.json to use Arabic dataset ---
cfg_path = f'{REPO}/config/train.json'
with open(cfg_path) as f:
    cfg = json.load(f)

cfg['dataset'] = {
    'name': 'Jr23xd23/ArabicText-Large',
    'split': 'train',
    'max_docs': 200000,
    'min_len': 50,
    'train_split': 0.95
}

with open(cfg_path, 'w') as f:
    json.dump(cfg, f, indent=4, ensure_ascii=False)

print(f'Config patched: dataset = Jr23xd23/ArabicText-Large')

In [ ]:
# 5. Patch instruct.json for Arabic finetuning (CIDAR)
import json

REPO = '/content/pg'

cfg_path = f'{REPO}/config/instruct.json'
with open(cfg_path) as f:
    cfg = json.load(f)

cfg['dataset']['alpaca_name'] = 'arbml/CIDAR'
cfg['dataset']['alpaca_max_output_len'] = 2000

with open(cfg_path, 'w', encoding='utf-8') as f:
    json.dump(cfg, f, indent=4, ensure_ascii=False)

print(f'Finetune config patched: dataset -> arbml/CIDAR (10K human-reviewed Arabic pairs)')
print(f'manual_data.json already in repo has 254 Arabic Q&A pairs')

In [ ]:
# 6. Start pretraining in the background
import subprocess, threading, time, shutil, os

REPO    = '/content/pg'
DRIVE_W = '/content/drive/MyDrive/pg-weights'

logf = open(f'{REPO}/pretrain.out', 'w')
proc = subprocess.Popen(
    ['python3', 'training/pretrain.py'],
    cwd=REPO, stdout=logf, stderr=subprocess.STDOUT
)
print(f'Pretraining started (PID {proc.pid})')

def sync_to_drive():
    while True:
        try:
            for f in ['pretrain_best.pt', 'log.jsonl']:
                src = f'{REPO}/weights/{f}'
                if os.path.exists(src):
                    shutil.copy(src, f'{DRIVE_W}/{f}')
        except Exception as e:
            print('sync error:', e)
        time.sleep(30)

threading.Thread(target=sync_to_drive, daemon=True).start()
print('Drive sync active (every 30s) - weights are safe')

In [ ]:
# 7. Watch pretraining progress (re-run anytime)
import subprocess
print(subprocess.run(['tail', '-15', '/content/pg/pretrain.out'],
                     capture_output=True, text=True).stdout)

In [ ]:
# 8. Wait for pretraining to finish, then start finetuning
import subprocess, time, shutil, os, threading

REPO    = '/content/pg'
DRIVE_W = '/content/drive/MyDrive/pg-weights'

# Wait for pretrain process to finish
while proc.poll() is None:
    time.sleep(10)
print(f'Pretraining finished (exit code {proc.returncode})')

# Sync final pretrain checkpoint
for f in ['pretrain_best.pt', 'log.jsonl']:
    src = f'{REPO}/weights/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_W}/{f}')

# Start finetuning
logf2 = open(f'{REPO}/finetune.out', 'w')
proc2 = subprocess.Popen(
    ['python3', 'training/finetune.py'],
    cwd=REPO, stdout=logf2, stderr=subprocess.STDOUT
)
print(f'Finetuning started (PID {proc2.pid})')

def sync_finetune_to_drive():
    while True:
        try:
            for f in ['instruct_best.pt', 'instruct_log.jsonl']:
                src = f'{REPO}/weights/{f}'
                if os.path.exists(src):
                    shutil.copy(src, f'{DRIVE_W}/{f}')
        except Exception as e:
            print('sync error:', e)
        time.sleep(30)

threading.Thread(target=sync_finetune_to_drive, daemon=True).start()
print('Finetune Drive sync active')

In [ ]:
# 9. Watch finetuning progress (re-run anytime)
import subprocess
print(subprocess.run(['tail', '-15', '/content/pg/finetune.out'],
                     capture_output=True, text=True).stdout)

In [ ]:
# 10. Check what's saved in Drive
import os
DRIVE_W = '/content/drive/MyDrive/pg-weights'
for f in sorted(os.listdir(DRIVE_W)):
    p = os.path.join(DRIVE_W, f)
    print(f'{f:30s} {os.path.getsize(p)/1e6:.1f} MB')
print('\nCheckpoints in Drive = your safe weights')

In [ ]:
# 11. Test the finetuned model
import sys
sys.path.insert(0, '/content/pg/llm')
from generate import chat

for q in ["ما اسمك؟", "ما هو 2 + 2؟", "ما هي عاصمة فرنسا؟", "أخبرني قصة قصيرة."]:
    print(f'Q: {q}')
    print(f'A: {chat(q)}')
    print('-' * 40)